<a href="https://colab.research.google.com/github/kjahan/armory/blob/main/notebooks/thoth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Thoth: Download, Summarize, Paraphraze, Publish, Repeat!

`
Thoth was the god of the moon, sacred texts, mathematics, the sciences, magic, messenger and recorder of the deities, master of knowledge, and patron of scribes. His Egyptian name was Djehuty, which means “He who is like the Ibis.” He was depicted as an ibis bird or a baboon.
`

1. Take a web page and extract its text
2. Pass text one paragraph at a time to summarizer
3. Paraphrase each paragraph
4. Add paraphrased paragraphs back together
5. Final Edit
6. Publish them!

## Instal Transfomers

In [1]:
# !pip install transformers
!pip install transformers[sentencepiece]

     |████████████████████████████████| 3.4 MB 11.2 MB/s 
     |████████████████████████████████| 596 kB 49.8 MB/s 
     |████████████████████████████████| 61 kB 430 kB/s 
     |████████████████████████████████| 895 kB 50.3 MB/s 
     |████████████████████████████████| 3.3 MB 43.1 MB/s 
     |████████████████████████████████| 1.2 MB 41.2 MB/s 
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 3.13
    Uninstalling PyYAML-3.13:
      Successfully uninstalled PyYAML-3.13


## Imports

In [22]:
import requests
from bs4 import BeautifulSoup

from random import sample

import torch

from transformers import pipeline
from transformers import PegasusForConditionalGeneration, PegasusTokenizer

## Web Scraper

In [3]:
def scrape(url):    
    headers = {
        'authority': 'www.amazon.com',
        'pragma': 'no-cache',
        'cache-control': 'no-cache',
        'dnt': '1',
        'upgrade-insecure-requests': '1',
        'user-agent': 'Mozilla/5.0 (X11; CrOS x86_64 8172.45.0) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/51.0.2704.64 Safari/537.36',
        'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9',
        'sec-fetch-site': 'none',
        'sec-fetch-mode': 'navigate',
        'sec-fetch-dest': 'document',
        'accept-language': 'en-GB,en-US;q=0.9,en;q=0.8',
    }

    # scrape the page using requests
    print("Scraping {}".format(url))
    r = requests.get(url, headers=headers)
    # check if page was blocked (Usually 503)
    if r.status_code > 500:
        if "To discuss automated access to Amazon data please contact" in r.text:
            print("Page {} was blocked by Amazon. Please try using better proxies".format(url))
        else:
            print("Page {} must have been blocked by Amazon as the status code was {}".format(url,r.status_code))
        return None
    # pass the HTML of the page 
    return r.text

## Step 1: Scrape

Article title:
`Travel stocks fall as Omicron spurs mass flight cancellations for fourth day`

https://www.reuters.com/markets/europe/rising-omicron-cases-disrupt-air-travel-800-more-flights-canceled-2021-12-27/

In [4]:
url = "https://www.reuters.com/markets/europe/rising-omicron-cases-disrupt-air-travel-800-more-flights-canceled-2021-12-27/"

html = scrape(url)

Scraping https://www.reuters.com/markets/europe/rising-omicron-cases-disrupt-air-travel-800-more-flights-canceled-2021-12-27/


## Review scraped page

In [5]:
# Make sure article title is in the craped html page
title = "Travel stocks fall as Omicron spurs mass flight cancellations for fourth day"

assert title in html
assert len(html) > 0

## Step 2: Parse web page & extract paragraphs

In [7]:
soup = BeautifulSoup(html)

p_elems = soup.findAll('p')

# store all article paragraphs
paragraphs = []

for paragraph in p_elems:
  # print(paragraph.text)
  paragraphs.append(paragraph.text)

## Total length of all paragraphs in chars

In [8]:
original_len_chars = 0

for paragraph in paragraphs:
  original_len_chars += len(paragraph)

## Review

In [9]:
print("No of paragraphs in original article: {}".format(len(paragraphs)))

assert len(paragraphs) > 0
assert len(paragraphs[0]) > 0

No of paragraphs in original article: 28


## Tokenizer

In [10]:
def get_tokens(text):
  tokens = text.split()
  return tokens

## Step 3: Merge short paragraphs befor summarization step

In [11]:
longer_paragraphs = []

max_token_threshold = 100
merged_paragraphs = []
current_tokens_no = 0

for inx in range(len(paragraphs)):
  cur_paragraph = paragraphs[inx]
  tokens = get_tokens(cur_paragraph)
  if not merged_paragraphs:
    # first paragraph to be added
    # print("Start state --> paragraph: {}".format(cur_paragraph))
    merged_paragraphs.append(cur_paragraph)
    current_tokens_no += len(tokens)
  elif current_tokens_no + len(tokens) <= max_token_threshold:
    # keep merging
    # print("keep merging --> current_tokens_no: {}".format(current_tokens_no))
    merged_paragraphs.append(cur_paragraph)
    current_tokens_no += len(tokens)
  else:
    # Done merging
    # print("Reset --> current_tokens_no: {}".format(current_tokens_no))
    new_paragraph = " ".join(merged_paragraphs)
    longer_paragraphs.append(new_paragraph)
    # Reset state
    merged_paragraphs = []
    current_tokens_no = 0


## Review

In [12]:
print("No of longer paragraphs: {}".format(len(longer_paragraphs)))

assert len(longer_paragraphs) > 0
assert len(longer_paragraphs) <= len(paragraphs)

No of longer paragraphs: 5


## Step 4: Summarize --> BART

In [13]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

Downloading:   0%|          | 0.00/1.55k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.51G [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/878k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/446k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.29M [00:00<?, ?B/s]

## Process paragraph by pargraph and run summarizer

In [14]:
summaries = []

for paragraph in longer_paragraphs:
    summary = summarizer(paragraph, max_length=130, min_length=30, do_sample=False)
    summaries.append(summary)

Your max_length is set to 130, but you input_length is only 110. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=55)
Your max_length is set to 130, but you input_length is only 126. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=63)


## Test

In [15]:
assert len(summaries) == len(longer_paragraphs)

## Step 5: Run paraphraser

## Download PEGASUS model & set Device
Paraphraser using PEGASUS
PEGASUS fine-tuned for paraphrasing

Ref: https://huggingface.co/tuner007/pegasus_paraphrase

In [16]:
model_name = 'tuner007/pegasus_paraphrase'
torch_device = 'cuda' if torch.cuda.is_available() else 'cpu'

print("torch device: {}".format(torch_device))

tokenizer = PegasusTokenizer.from_pretrained(model_name)
model = PegasusForConditionalGeneration.from_pretrained(model_name).to(torch_device)

torch device: cuda


Downloading:   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/2.12G [00:00<?, ?B/s]

## Process Pegaus model output

In [17]:
def get_response(input_text, num_return_sequences, num_beams):
  batch = tokenizer([input_text], truncation=True, padding='longest', max_length=60, return_tensors="pt").to(torch_device)
  translated = model.generate(**batch, max_length=60, num_beams=num_beams, num_return_sequences=num_return_sequences, temperature=1.5)
  tgt_text = tokenizer.batch_decode(translated, skip_special_tokens=True)
  return tgt_text

## Test paraphraser

In [18]:
num_beams = 10
num_return_sequences = 10

context = "The ultimate test of your knowledge is your capacity to convey it to another."
get_response(context, num_return_sequences, num_beams)

['The test of your knowledge is your ability to convey it.',
 'The ability to convey your knowledge is the ultimate test of your knowledge.',
 'The ability to convey your knowledge is the most important test of your knowledge.',
 'Your capacity to convey your knowledge is the ultimate test of it.',
 'The test of your knowledge is your ability to communicate it.',
 'Your capacity to convey your knowledge is the ultimate test of your knowledge.',
 'Your capacity to convey your knowledge to another is the ultimate test of your knowledge.',
 'Your capacity to convey your knowledge is the most important test of your knowledge.',
 'The test of your knowledge is how well you can convey it.',
 'Your capacity to convey your knowledge is the ultimate test.']

## Run paraphraser

In [21]:
num_beams = 10
num_return_sequences = 10

paraphrased_candidates = []

for item in summaries:
  context = item[0]['summary_text']
  candidates = get_response(context, num_return_sequences, num_beams)
  paraphrased_candidates.append(candidates)

## Review

In [ ]:
assert len(paraphrased_candidates) == len(summaries)
assert len(paraphrased_candidates[0]) == num_return_sequences

## Sample Paraphrased paragraphs & Stich them together

In [24]:
pieces = []

for candidates in paraphrased_candidates:
  candidate = sample(candidates, 1)[0]
  # print(candidate)
  pieces.append(candidate)

overall_summary = " ".join(pieces)

## Summary Ratio

In [25]:
print("Summary ratio: {}".format(1.0*len(overall_summary)/original_len_chars))

Summary ratio: 0.11515552614162806


## Publish

In [26]:
print(overall_summary)

On Monday, over 1,000 flights were canceled in the US. Staff shortages at airlines, weather-related disruptions, and now the Omicron variant have disrupted flights frequently this year. American Airlines says it had to cancel flights due to "COVID-related sick calls." The China Eastern Airlines flights were suspended from New York to Shanghai. The Carnival Freedom cruise ship had a small number of passengers isolated due to positive test results. The most complete solution to manage all your tax and compliance needs.
